# Extract Occupations and Skills from PAD Chunks
Extract occupations and skills from markdown chunks using OpenAI custom GPT.

**Note:** The reusable occupation extraction logic has been implemented in `src/extraction/`. Use the CLI for production:
- Extract occupations: `uv run python -m src.extraction.cli_occupations`

This notebook contains the original exploration and can be used for testing individual projects.

## 1. Import Required Libraries

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

from openai import OpenAI

# Import our config
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config

## 2. Load Configuration and Environment Variables

In [2]:
# Load environment variables from .env file
project_root = Path.cwd().parent
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"'.env' file not found at {env_path}\n"
        "Please copy .env.example to .env and add your OpenAI API key."
    )

# Load from specific path
load_dotenv(env_path, override=True)

# Load project config
config = load_config()

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify API key is set
if not OPENAI_API_KEY:
    raise ValueError("Missing required environment variable: OPENAI_API_KEY")

print("✓ Environment variables loaded")
print(f"  API Key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")

✓ Environment variables loaded
  API Key: sk-proj-cj...__0A


## 3. Set Up Paths

In [3]:
# Get paths
md_dir = project_root / config.paths.markdown
chunks_dir = md_dir.parent / "pads_md_chunks"
output_dir = project_root / "data" / "silver" / "occupations_skills_json"

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Chunks directory: {chunks_dir}")
print(f"Output directory: {output_dir}")
print(f"Chunks exist: {chunks_dir.exists()}")

Chunks directory: /Users/lauren/repos/PAD2Skills/data/silver/pads_md_chunks
Output directory: /Users/lauren/repos/PAD2Skills/data/silver/occupations_skills_json
Chunks exist: True


In [4]:
project_id = "P511453"  # Example project ID

## 4. Load Chunk Files

In [5]:
# Find all markdown chunk files
chunk_files = sorted(chunks_dir.glob(f"{project_id}_*.md"))

print(f"Found {len(chunk_files)} chunk files")
print("\nFirst 5 chunks:")
for chunk_file in chunk_files[:5]:
    size_kb = chunk_file.stat().st_size / 1024
    print(f"  {chunk_file.name:60s} {size_kb:6.2f} KB")

if len(chunk_files) > 5:
    print(f"  ... and {len(chunk_files) - 5} more")

Found 8 chunk files

First 5 chunks:
  P511453_0_strategic_context.md                                21.85 KB
  P511453_1_project_description.md                              58.74 KB
  P511453_2_project_implementation.md                            8.96 KB
  P511453_3_project_appraisal_summary.md                        31.32 KB
  P511453_4_key_risks.md                                       259.73 KB
  ... and 3 more


## 5. Initialize OpenAI Client

In [6]:
# Initialize OpenAI client
client = OpenAI()

print("✓ OpenAI client initialized")

✓ OpenAI client initialized


## 6. Load PAD Summary

In [7]:
# Determine project_id from first chunk file
if chunk_files:
    first_chunk = chunk_files[0]
    project_id = first_chunk.stem.split('_', 1)[0]
    
    # Load PAD summary file for this project
    summary_dir = project_root / "data" / "silver" / "pad_summaries"
    summary_file = summary_dir / f"{project_id}_summary.txt"
    
    if summary_file.exists():
        pad_summary = summary_file.read_text(encoding='utf-8').strip()
        print(f"✓ Loaded PAD summary: {summary_file.name}")
        print(f"  Size: {len(pad_summary)} chars")
        print(f"  Preview:\n{pad_summary[:200]}...")
    else:
        pad_summary = ""
        print(f"⚠ No PAD summary file found: {summary_file}")
        print("  Proceeding without PAD summary context")
else:
    pad_summary = ""
    print("⚠ No chunk files found")

✓ Loaded PAD summary: P511453_summary.txt
  Size: 1932 chars
  Preview:
The Guinea Electricity Access Scale Up Project-Phase 2 (GNEAP-2) will expand electricity access in selected urban and rural areas of Guinea. It is a US$271.8 million investment project financing with ...


## 7. Load Abbreviations File

In [10]:
# Load abbreviations file for this project
abbr_dir = project_root / "data" / "silver" / "abbreviations_md"
abbr_file = abbr_dir / f"{project_id}_1_abbr.md"

if abbr_file.exists():
    abbreviations_text = abbr_file.read_text(encoding='utf-8')
    print(f"✓ Loaded abbreviations file: {abbr_file.name}")
    print(f"  Size: {len(abbreviations_text)} chars")
    print(f"  Preview:\n{abbreviations_text[:200]}...")
else:
    abbreviations_text = ""
    print(f"⚠ No abbreviations file found: {abbr_file}")
    print("  Proceeding without abbreviations context")

✓ Loaded abbreviations file: P511453_1_abbr.md
  Size: 3649 chars
  Preview:
Abbreviation | Definition
--- | ---
AFD | Agence Française de Développement (French Development Agency)
AfDB | African Development Bank
AGEE | Agence Guinéenne d'Evaluations Environnementales (Guinean...


## 8. Process Each Chunk

In [11]:
print(f"Processing {len(chunk_files)} chunks...")
print()

processed_chunks = []

for i, chunk_file in enumerate(chunk_files, 1):
    # Parse filename: {project_id}_{section_id}_{snake_title}.md
    filename_parts = chunk_file.stem.split('_', 2)
    project_id = filename_parts[0]
    section_id = filename_parts[1]
    
    # Read chunk content
    chunk_text = chunk_file.read_text(encoding='utf-8')
    
    # Prepend abbreviations if available
    if abbreviations_text:
        chunk_text_with_context = abbreviations_text + "\n\n" + chunk_text
    else:
        chunk_text_with_context = chunk_text
    
    print(f"[{i}/{len(chunk_files)}] Processing: {chunk_file.name}")
    print(f"  Project ID: {project_id}, Section ID: {section_id}")
    print(f"  Chunk size: {len(chunk_text)} chars")
    if abbreviations_text:
        print(f"  With abbreviations: {len(chunk_text_with_context)} chars")
    
    # Prepare input for custom GPT with project_summary
    input_message = f"project_id: {project_id}\nsection_id: {section_id}\nproject_summary: {pad_summary}\nchunk_text: {chunk_text_with_context}"
    
    # Call custom GPT with prompt ID
    response = client.responses.create(
        prompt={
            "id": "pmpt_6950c224bab0819486a7f38e0ae0109b08192593c3d4b4af",
            "version": "18"
        },
        input=[
            {"role": "user", "content": input_message}
        ],
        reasoning={
            "summary": None
        },
        store=False,
        include=[
            "reasoning.encrypted_content",
            "web_search_call.action.sources"
        ]
    )
    
    # Extract the text from the response
    result = None
    for item in response.output:
        if hasattr(item, 'content') and hasattr(item, 'role'):
            result = item.content[0].text
            break
    
    # Save result to file
    output_file = output_dir / f"{project_id}_{section_id}_occupations.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(result)
    
    processed_chunks.append({
        'chunk_file': chunk_file.name,
        'project_id': project_id,
        'section_id': section_id,
        'output_file': output_file.name,
        'response_id': response.id
    })
    
    print(f"  ✓ Saved to: {output_file.name}")
    print()

print("=" * 80)
print(f"✓ Processed {len(processed_chunks)} chunks")
print(f"✓ Results saved to: {output_dir}")

Processing 8 chunks...

[1/8] Processing: P511453_0_strategic_context.md
  Project ID: P511453, Section ID: 0
  Chunk size: 22352 chars
  With abbreviations: 26003 chars
  ✓ Saved to: P511453_0_occupations.json

[2/8] Processing: P511453_1_project_description.md
  Project ID: P511453, Section ID: 1
  Chunk size: 60100 chars
  With abbreviations: 63751 chars
  ✓ Saved to: P511453_1_occupations.json

[3/8] Processing: P511453_2_project_implementation.md
  Project ID: P511453, Section ID: 2
  Chunk size: 9173 chars
  With abbreviations: 12824 chars
  ✓ Saved to: P511453_2_occupations.json

[4/8] Processing: P511453_3_project_appraisal_summary.md
  Project ID: P511453, Section ID: 3
  Chunk size: 32066 chars
  With abbreviations: 35717 chars
  ✓ Saved to: P511453_3_occupations.json

[5/8] Processing: P511453_4_key_risks.md
  Project ID: P511453, Section ID: 4
  Chunk size: 265803 chars
  With abbreviations: 269454 chars
  ✓ Saved to: P511453_4_occupations.json

[6/8] Processing: P511453_5_

## 8. Verify Output Files

In [12]:
# List all output files
output_files = sorted(output_dir.glob("*_occupations.json"))

print(f"Created {len(output_files)} output files:")
print("=" * 80)

for output_file in output_files[:10]:  # Show first 10
    size_kb = output_file.stat().st_size / 1024
    print(f"  {output_file.name:60s} {size_kb:6.2f} KB")

if len(output_files) > 10:
    print(f"  ... and {len(output_files) - 10} more")

print("=" * 80)
print(f"Total output files: {len(output_files)}")

Created 133 output files:
  P075941_0_occupations.json                                     3.71 KB
  P075941_10_occupations.json                                    7.83 KB
  P075941_11_occupations.json                                    3.57 KB
  P075941_12_occupations.json                                    4.78 KB
  P075941_13_occupations.json                                    9.35 KB
  P075941_14_occupations.json                                    6.42 KB
  P075941_15_occupations.json                                    0.07 KB
  P075941_1_occupations.json                                     4.05 KB
  P075941_2_occupations.json                                    21.74 KB
  P075941_3_occupations.json                                    10.38 KB
  ... and 123 more
Total output files: 133
